# CardioIA – Fase 2 | Parte 1: Frases de sintomas + extração de informações

Este notebook lê os relatos de pacientes (`frases_sintomas.txt`), identifica sintomas com base no **mapa de conhecimento** (`mapa_conhecimento.csv`) e sugere um possível diagnóstico.

> ⚠️ Projeto **acadêmico**. As sugestões são baseadas em regras simples e **não substituem avaliação médica**.

**Como funciona (resumo):**
1. Lê as 10 frases e o mapa de conhecimento (`Sintoma 1 | Sintoma 2 | Doença Associada`).
2. Normaliza o texto (minúsculas e sem acentos) para que `Palpitações` e `palpitacoes` sejam equivalentes.
3. Procura cada sintoma do mapa dentro da frase.
4. Para cada doença, conta **quantos sintomas distintos** foram encontrados; a doença com mais evidências é a sugestão principal e as demais aparecem como alternativas.

In [1]:
import unicodedata
import pandas as pd

def normalizar(texto):
    """Minúsculas e sem acentos, para comparar textos de forma robusta."""
    texto = unicodedata.normalize("NFKD", str(texto).lower())
    return "".join(c for c in texto if not unicodedata.combining(c)).strip()

# 1) Relatos dos pacientes (ignorando linhas vazias)
with open("frases_sintomas.txt", "r", encoding="utf-8") as arquivo:
    frases = [linha.strip() for linha in arquivo if linha.strip()]

print(f"{len(frases)} relatos carregados:\n")
for i, frase in enumerate(frases, start=1):
    print(f"{i:>2} - {frase}")

10 relatos carregados:

 1 - Há dois dias estou com uma forte dor no peito que piora quando faço esforço físico.
 2 - Há uma semana sinto cansaço constante e tenho dificuldade para realizar minhas atividades diárias.
 3 - Desde ontem estou sentindo falta de ar e dificuldade para respirar quando caminho.
 4 - Há três dias sinto palpitações e percebo meu coração batendo muito rápido mesmo quando estou parado.
 5 - Desde esta manhã estou com uma sensação de aperto no peito que aparece principalmente quando subo escadas.
 6 - Há quatro dias estou sentindo tontura e fraqueza, principalmente quando me levanto rapidamente.
 7 - Há uma semana percebo inchaço nas pernas e muito cansaço ao caminhar por alguns minutos.
 8 - Desde ontem sinto uma forte dor no peito acompanhada de suor frio e isso está me impedindo de realizar minhas atividades normalmente.
 9 - Há alguns dias sinto meu coração bater de maneira irregular e isso acontece várias vezes durante o dia.
10 - Há duas semanas sinto muito c

## 1. Mapa de conhecimento (sintoma → doença)

In [2]:
# 2) Mapa de conhecimento: colunas separadas por ';'
mapa = pd.read_csv("mapa_conhecimento.csv", sep=";", encoding="utf-8-sig")
print(f"{len(mapa)} regras | doenças no mapa: {sorted(mapa['Doença Associada'].unique())}")
display(mapa)

37 regras | doenças no mapa: ['Angina', 'Arritmia', 'Hipertensão', 'Hipotensão', 'Infarto', 'Insuficiência Cardíaca', 'Taquicardia']


                     Sintoma 1                     Sintoma 2        Doença Associada
0                 dor no peito                     suor frio                 Infarto
1                 dor no peito         dor no braço esquerdo                 Infarto
2                 dor no peito                        náusea                 Infarto
3             pressão no peito                     suor frio                 Infarto
4                 dor no peito          irradia para o braço                 Infarto
5                 dor no peito              dor na mandíbula                 Infarto
6                 dor no peito                       esforço                  Angina
7              aperto no peito                       esforço                  Angina
8              aperto no peito                 subir escadas                  Angina
9              aperto no peito                  subo escadas                  Angina
10                dor no peito                   falta de ar     

## 2. Função de extração e sugestão de diagnóstico

Pontos importantes da lógica:
- Todos os sintomas encontrados contam, **somando por doença** (um sintoma que aparece em várias regras da mesma doença conta só uma vez).
- Se um sintoma curto está dentro de outro mais longo já encontrado (ex.: `cansaço` dentro de `cansaço constante`), conta apenas o mais específico.
- Em caso de empate, todas as doenças empatadas são mostradas.

In [3]:
def extrair_sintomas(frase, mapa):
    """Retorna {doença: [sintomas encontrados]} para uma frase."""
    texto = normalizar(frase)

    # todos os sintomas distintos do mapa e a(s) doença(s) a que pertencem
    ligacoes, nome_original = {}, {}
    for _, linha in mapa.iterrows():
        for coluna in ("Sintoma 1", "Sintoma 2"):
            sintoma = normalizar(linha[coluna])
            nome_original[sintoma] = linha[coluna]          # guarda a grafia original (com acento)
            ligacoes.setdefault(sintoma, set()).add(linha["Doença Associada"])

    # sintomas presentes na frase
    achados = [s for s in ligacoes if s in texto]
    # descarta os que estão contidos em outro sintoma mais específico também achado
    achados = [s for s in achados if not any(s != outro and s in outro for outro in achados)]

    por_doenca = {}
    for sintoma in achados:
        for doenca in ligacoes[sintoma]:
            por_doenca.setdefault(doenca, []).append(nome_original[sintoma])
    return por_doenca


def sugerir_diagnostico(frase, mapa):
    """Retorna (doenças_principais, sintomas_da_principal, ranking_completo)."""
    por_doenca = extrair_sintomas(frase, mapa)
    if not por_doenca:
        return [], [], []

    ranking = sorted(por_doenca.items(), key=lambda item: len(item[1]), reverse=True)
    melhor = len(ranking[0][1])
    principais = [d for d, s in ranking if len(s) == melhor]
    sintomas = sorted({s for d, ss in ranking if d in principais for s in ss})
    return principais, sintomas, ranking

## 3. Teste com uma frase

In [4]:
frase_teste = "Desde ontem sinto uma forte dor no peito acompanhada de suor frio."
principais, sintomas, ranking = sugerir_diagnostico(frase_teste, mapa)

print("Relato:", frase_teste)
print("Sintomas identificados:", ", ".join(sintomas))
print("Possível diagnóstico:", " ou ".join(principais))
print("Ranking completo:", [(d, len(s)) for d, s in ranking])

Relato: Desde ontem sinto uma forte dor no peito acompanhada de suor frio.
Sintomas identificados: dor no peito, suor frio
Possível diagnóstico: Infarto
Ranking completo: [('Infarto', 2), ('Angina', 1)]


## 4. Análise de todos os pacientes

In [5]:
print("=== CARDIOIA - ANÁLISE DOS PACIENTES ===\n")
resumo = []

for numero, frase in enumerate(frases, start=1):
    principais, sintomas, ranking = sugerir_diagnostico(frase, mapa)
    diagnostico = " ou ".join(principais) if principais else "Nenhum diagnóstico encontrado"
    alternativas = [d for d, s in ranking if d not in principais]

    print(f"PACIENTE {numero}")
    print("Relato:", frase)
    print("Sintomas identificados:", ", ".join(sintomas) if sintomas else "nenhum")
    print("Possível diagnóstico:", diagnostico)
    if alternativas:
        print("Outras hipóteses:", ", ".join(alternativas))
    print("-" * 60)

    resumo.append({"Paciente": numero, "Sintomas": ", ".join(sintomas), "Diagnóstico sugerido": diagnostico})

=== CARDIOIA - ANÁLISE DOS PACIENTES ===

PACIENTE 1
Relato: Há dois dias estou com uma forte dor no peito que piora quando faço esforço físico.
Sintomas identificados: dor no peito, esforço
Possível diagnóstico: Angina
Outras hipóteses: Infarto
------------------------------------------------------------
PACIENTE 2
Relato: Há uma semana sinto cansaço constante e tenho dificuldade para realizar minhas atividades diárias.
Sintomas identificados: cansaço constante, dificuldade para realizar
Possível diagnóstico: Insuficiência Cardíaca
------------------------------------------------------------
PACIENTE 3
Relato: Desde ontem estou sentindo falta de ar e dificuldade para respirar quando caminho.
Sintomas identificados: dificuldade para respirar, falta de ar
Possível diagnóstico: Insuficiência Cardíaca
Outras hipóteses: Angina, Taquicardia
------------------------------------------------------------
PACIENTE 4
Relato: Há três dias sinto palpitações e percebo meu coração batendo muito rápid

In [6]:
# Tabela-resumo
display(pd.DataFrame(resumo))

   Paciente                                           Sintomas    Diagnóstico sugerido
0         1                              dor no peito, esforço                  Angina
1         2       cansaço constante, dificuldade para realizar  Insuficiência Cardíaca
2         3             dificuldade para respirar, falta de ar  Insuficiência Cardíaca
3         4          coração batendo muito rápido, palpitações             Taquicardia
4         5                      aperto no peito, subo escadas                  Angina
5         6             fraqueza, levanto rapidamente, tontura              Hipotensão
6         7                        cansaço, inchaço nas pernas  Insuficiência Cardíaca
7         8                            dor no peito, suor frio                 Infarto
8         9                         bater de maneira irregular                Arritmia
9        10  cansaço, dormir quando estou deitado, falta de ar  Insuficiência Cardíaca

## 5. Limitações

- A busca é por **trechos de texto**: se o paciente usar uma expressão que não está no mapa (ex.: "coração disparando"), o sintoma não é reconhecido. Quanto mais linhas no mapa, melhor o resultado.
- **Negações** não são tratadas: "não sinto dor no peito" ainda encontraria `dor no peito`.
- O mapa é uma simplificação didática, **não** um protocolo clínico. Sintomas como "dor no peito" são compartilhados por várias doenças e exigem exames e avaliação médica.
- Este tipo de sistema deve apenas **apoiar** o profissional de saúde, nunca substituí-lo (governança de dados e uso responsável de IA).